In [41]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
sc.settings.verbosity = 3

# Instead of set_figure_params, configure matplotlib manually
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.dpi'] = 100


In [42]:
%matplotlib inline

In [43]:
def run_and_plot_umap(
    adata,
    use_rep,
    color=["cell_type_level1","batch"],
    save_dir="umap_plots",
    title_prefix=None,
    figsize=(10, 8),
    dpi=100,
    dot_size=3 
):
    os.makedirs(save_dir, exist_ok=True)

    # Compute neighbors + UMAP
    sc.pp.neighbors(adata, use_rep=use_rep)
    sc.tl.umap(adata)

    # Defaults
    title_prefix = title_prefix if title_prefix else "UMAP"

    # Create figure
    plt.figure(figsize=figsize)
    sc.pl.umap(
        adata,
        color=color,
        title=f"{title_prefix} from {use_rep}",
        show=False,
        frameon=True,
        s=dot_size  # set point size
    )

    # Save
    save_path = os.path.join(save_dir, f"{title_prefix}.png")
    plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"Saved {save_path}")

In [44]:
combined_unharm_sampled = sc.read_h5ad("combined_unharm_sampled.h5ad")
combined_harm_sampled = sc.read_h5ad("combined_harm_sampled.h5ad")
combined_unharm_scetm_sampled = sc.read_h5ad("combined_unharm_scetm_sampled.h5ad")


In [45]:
combined_unharm_sampled

AnnData object with n_obs × n_vars = 250000 × 5634
    obs: 'total_counts', 'cell_type_level1', 'batch', 'batch_indices', 'cell_types'
    uns: 'batch_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama', 'X_scetm', 'X_umap', 'theta'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [46]:
combined_harm_sampled

AnnData object with n_obs × n_vars = 250000 × 5634
    obs: 'total_counts', 'cell_type_level1', 'batch'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [47]:
combined_unharm_scetm_sampled

AnnData object with n_obs × n_vars = 250000 × 50
    obs: 'total_counts', 'cell_type_level1', 'batch', 'batch_indices', 'cell_types'
    uns: 'batch_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama'

In [48]:
n_samples = 50000   # number of cells per batch
keep_indices = []

for b in combined_unharm_sampled.obs["batch"].unique():
    batch_indices = np.where(combined_unharm_sampled.obs["batch"] == b)[0]
    n_keep = min(n_samples, len(batch_indices))  # in case batch has fewer cells
    sampled = np.random.choice(batch_indices, n_keep, replace=False)
    keep_indices.extend(sampled)



In [49]:
combined_unharm_sampled = combined_unharm_sampled[keep_indices].copy()
combined_harm_sampled = combined_harm_sampled[keep_indices].copy()
combined_unharm_scetm_sampled = combined_unharm_scetm_sampled[keep_indices].copy()

In [50]:
combined_unharm_sampled

AnnData object with n_obs × n_vars = 100000 × 5634
    obs: 'total_counts', 'cell_type_level1', 'batch', 'batch_indices', 'cell_types'
    uns: 'batch_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama', 'X_scetm', 'X_umap', 'theta'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [51]:
combined_harm_sampled

AnnData object with n_obs × n_vars = 100000 × 5634
    obs: 'total_counts', 'cell_type_level1', 'batch'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [52]:
combined_unharm_scetm_sampled

AnnData object with n_obs × n_vars = 100000 × 50
    obs: 'total_counts', 'cell_type_level1', 'batch', 'batch_indices', 'cell_types'
    uns: 'batch_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama'

Unharmonized UMAP

In [ ]:
run_and_plot_umap(combined_unharm_sampled,"X_pca",title_prefix="unharmonized")


Harmonised UMAP

In [ ]:
run_and_plot_umap(combined_harm_sampled,"X_pca",title_prefix="harmonized")


Unharmonized_scETM UMAP

In [ ]:
run_and_plot_umap(combined_unharm_scetm_sampled,"X_pca",title_prefix="unharmonized_scetm")

Scaranoma Unharmoinzed

In [ ]:
run_and_plot_umap(combined_unharm_sampled,"X_scanorama",title_prefix="unharmonized_scanorama")

Scaranoma Unharmonized scETM

In [ ]:
run_and_plot_umap(combined_unharm_scetm_sampled,"X_scanorama",title_prefix="unharmonized_scetm_scanorama")

Harmony Unharmonized

In [ ]:
run_and_plot_umap(combined_unharm_sampled,"X_scanorama",title_prefix="unharmonized_scanorama")

Harmony Unharmonized scETM

In [ ]:
run_and_plot_umap(combined_unharm_scetm_sampled,"X_scanorama",title_prefix="unharmonized_scetm_scanorama")

scVI Unharmonized

In [ ]:
run_and_plot_umap(combined_unharm_sampled,"X_scvi",title_prefix="unharmonized_scvi")

scVI unharmonized scETM 

In [ ]:
run_and_plot_umap(combined_unharm_scetm_sampled,"X_scvi",title_prefix="unharmonized_scetm_scvi")

Metrics


kBET

In [61]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.stats import chi2_contingency

def kbet_chi2(embedding, batch_labels, k=15, alpha=0.05):

    n = embedding.shape[0]
    nn = NearestNeighbors(n_neighbors=k+1, n_jobs=-1).fit(embedding)
    neighbors = nn.kneighbors(return_distance=False)
    
    batch_labels = np.array(batch_labels)
    unique_batches = np.unique(batch_labels)
    global_counts = np.bincount(batch_labels, minlength=len(unique_batches))
    global_props = global_counts / n

    rejections = 0

    for i in range(n):
        # Get neighbors excluding self
        neigh_idx = neighbors[i][1:]
        neigh_batches = batch_labels[neigh_idx]
        local_counts = np.bincount(neigh_batches, minlength=len(unique_batches))
        
        # Compute expected counts
        expected = global_props * local_counts.sum()
        
        # Chi-squared test
        chi2, p, dof, ex = chi2_contingency([local_counts, expected], correction=False)
        
        if p < alpha:
            rejections += 1
    
    # kBET score = fraction of neighborhoods NOT rejected
    kBET_score = 1 - (rejections / n)
    return kBET_score



In [63]:
selected_keys = ["X_pca", "X_pca_harmony", "X_scanorama"]
results = {}

# Encode batch labels as integers
batch_labels = combined_unharm_sampled.obs["batch"].astype("category").cat.codes.values

for key in selected_keys:
    print(f"Running kBET for {key} ...")
    emb = combined_unharm_sampled.obsm[key]
    score = kbet_chi2(emb, batch_labels, k=15, alpha=0.05)
    results[key] = score

print("\nKBET scores (higher = better batch mixing):")
for k, v in results.items():
    print(f"{k}: {v:.3f}")

Running kBET for X_pca ...
Running kBET for X_pca_harmony ...
Running kBET for X_scanorama ...

KBET scores (higher = better batch mixing):
X_pca: 0.001
X_pca_harmony: 0.033
X_scanorama: 0.001


In [62]:

selected_keys = ["X_pca", "X_pca_harmony", "X_scanorama"]
results = {}

# Encode batch labels as integers
batch_labels = combined_unharm_scetm_sampled.obs["batch"].astype("category").cat.codes.values

for key in selected_keys:
    print(f"Running kBET for {key} ...")
    emb = combined_unharm_scetm_sampled.obsm[key]
    score = kbet_chi2(emb, batch_labels, k=15, alpha=0.05)
    results[key] = score

print("\nKBET scores (higher = better batch mixing):")
for k, v in results.items():
    print(f"{k}: {v:.3f}")


Running kBET for X_pca ...
Running kBET for X_pca_harmony ...
Running kBET for X_scanorama ...

KBET scores (higher = better batch mixing):
X_pca: 0.012
X_pca_harmony: 0.010
X_scanorama: 0.030


In [64]:

selected_keys = ["X_pca"]
results = {}

# Encode batch labels as integers
batch_labels = combined_harm_sampled.obs["batch"].astype("category").cat.codes.values

for key in selected_keys:
    print(f"Running kBET for {key} ...")
    emb = combined_harm_sampled.obsm[key]
    score = kbet_chi2(emb, batch_labels, k=15, alpha=0.05)
    results[key] = score

print("\nKBET scores (higher = better batch mixing):")
for k, v in results.items():
    print(f"{k}: {v:.3f}")


Running kBET for X_pca ...

KBET scores (higher = better batch mixing):
X_pca: 0.234
